# JWCM v3: Bidirectional Jacobian-Weighted Criticality Merging

## Overview

This notebook implements **JWCM v3**, extending v2 with **bidirectional attribution**
to properly handle RLVR's dual nature: amplification AND suppression.

### Why v3 > v2 for RLVR

RLVR updates contain two types of signals:
- **Amplify (C+)**: increase probability of correct reasoning steps
- **Suppress (C-)**: decrease probability of wrong reasoning paths

In v2, if a parameter has opposing gradients from C+ and C- tokens,
they cancel in the aggregated gradient, making the parameter look unimportant:
- v2: $|g^+ + g^-| \cdot |\Delta\theta| = |0.5 + (-0.5)| = 0$ (hidden!)
- v3: $|g^+| \cdot |\Delta\theta| + |g^-| \cdot |\Delta\theta| = 0.5 + 0.5 = 1.0$ (captured!)

### Bidirectional Critical Sets
$$\mathcal{C}^+ = \{(x, y_t) : \Delta\log p(y_t) > +\epsilon\}$$
$$\mathcal{C}^- = \{(x, y_t) : \Delta\log p(y_t) < -\epsilon\}$$

### Attribution Scores
$$S_j^{\tau,+} = \sum_{(x,y_t) \in \mathcal{C}^+_\tau} |\nabla_{\theta_j} \log p(y_t) \cdot \Delta\theta_j^\tau|$$
$$S_j^{\tau,-} = \sum_{(x,y_t) \in \mathcal{C}^-_\tau} |\nabla_{\theta_j} \log p(y_t) \cdot \Delta\theta_j^\tau|$$
$$S_j^\tau = S_j^{\tau,+} + S_j^{\tau,-}$$

### Merge Formula (N-task scalable, same as v2)
$$\theta_j^{\text{merge}} = \theta_j^{\text{base}} + \frac{\sum_\tau S_j^\tau \cdot \Delta\theta_j^\tau}{\sum_\tau S_j^\tau + \epsilon}$$


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================================
# Configuration
# ============================================================================
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
MATH_MODEL_PATH = "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"

IF_DATA_PATH = "/mnt/ddn/vuvlm/geeho/datasets/Nemotron-Cascade-RL-IF/ifrl_final_release.parquet"
MATH_DATA_PATH = "/mnt/ddn/vuvlm/geeho/datasets/Nemotron-Cascade-RL-Math/math_verl_ready_MERGED_SYSTEM.parquet"

# Validation set size: 512 per benchmark
VAL_SAMPLES_PER_TASK = 512

# Generation settings
MAX_NEW_TOKENS = 512          # Max response length for generation
GENERATION_BATCH_SIZE = 16    # Batch size for greedy decoding

# Attribution settings
CRITICAL_PERCENTILE = 90      # Top 10% of |delta_log_p| are critical tokens
SPARSE_KEEP_RATIO = 0.20      # For v2+sparse: keep top 20% of parameters

# Reproducibility
SEED = 42
DTYPE = torch.float16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", context="talk")


In [ ]:
# ============================================================================
# Model and Data Utilities
# ============================================================================

def load_model(model_id_or_path: str, dtype=torch.float16, device: str = "cpu"):
    """Load a causal LM model.

    Args:
        model_id_or_path: HuggingFace model ID or local path.
        dtype: Weight dtype.
        device: Target device ("cpu" or "cuda").

    Returns:
        AutoModelForCausalLM instance.
    """
    model = AutoModelForCausalLM.from_pretrained(
        str(model_id_or_path),
        torch_dtype=dtype,
        device_map=device,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    return model


def load_tokenizer(model_id_or_path: str = BASE_MODEL_ID):
    """Load tokenizer with pad token configured for batch generation.

    Qwen3 models may not have an explicit pad token. We set it to eos_token
    so that left-padded batches work correctly during generation.

    Args:
        model_id_or_path: HuggingFace model ID or local path.

    Returns:
        AutoTokenizer instance.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        str(model_id_or_path),
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return tokenizer


def sample_validation_data(parquet_path: str, n: int, seed: int = 42) -> pd.DataFrame:
    """Sample n rows from a parquet file for validation.

    Args:
        parquet_path: Path to the parquet dataset.
        n: Number of samples to draw.
        seed: Random seed for reproducibility.

    Returns:
        Sampled DataFrame with reset index.
    """
    df = pd.read_parquet(parquet_path)
    if len(df) < n:
        print(f"Warning: dataset has only {len(df)} rows, using all.")
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)


def extract_prompts(df: pd.DataFrame, prompt_col: str = "prompt") -> List[List[dict]]:
    """Extract chat-format message lists from a DataFrame.

    The verl-ready parquet stores prompts as lists of dicts:
        [{"content": "...", "role": "user"}]

    Args:
        df: DataFrame with a prompt column.
        prompt_col: Column name containing chat messages.

    Returns:
        List of chat message lists.
    """
    prompts = []
    for _, row in df.iterrows():
        messages = row[prompt_col]
        if isinstance(messages, str):
            messages = json.loads(messages)
        prompts.append(messages)
    return prompts


def free_model(model, name: str = "model"):
    """Delete model and free GPU/CPU memory."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Freed {name}")


In [ ]:
# ============================================================================
# Generation and Log Probability Utilities
# ============================================================================

@torch.no_grad()
def generate_responses(
    model,
    tokenizer,
    prompts: List[List[dict]],
    max_new_tokens: int = 512,
    batch_size: int = 16,
) -> List[dict]:
    """Generate responses using greedy decoding (batch mode).

    Uses left-padding for decoder-only models so that the rightmost
    (most recent) tokens are always aligned.

    Args:
        model: Causal LM on GPU.
        tokenizer: Tokenizer with pad_token set.
        prompts: List of chat message lists.
        max_new_tokens: Maximum new tokens to generate.
        batch_size: Number of prompts per batch.

    Returns:
        List of dicts with keys: prompt_text, response_text,
        prompt_ids (LongTensor), response_ids (LongTensor).
    """
    model.eval()
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    results = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_msgs = prompts[i : i + batch_size]

        # Apply chat template to get prompt strings
        texts = []
        for msgs in batch_msgs:
            t = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
            texts.append(t)

        encoded = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model.device)

        # Prompt lengths (excluding left-padding)
        prompt_lengths = encoded.attention_mask.sum(dim=1).tolist()

        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        for j in range(len(batch_msgs)):
            p_len = int(prompt_lengths[j])
            # Account for left-padding offset
            pad_len = encoded.input_ids.shape[1] - p_len
            out = output_ids[j]

            prompt_ids = out[pad_len : pad_len + p_len].cpu()
            response_ids = out[pad_len + p_len :].cpu()
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

            results.append({
                "prompt_text": texts[j],
                "response_text": response_text,
                "prompt_ids": prompt_ids,
                "response_ids": response_ids,
            })

    tokenizer.padding_side = original_padding_side
    return results


@torch.no_grad()
def compute_token_log_probs(
    model,
    tokenizer,
    generated_results: List[dict],
) -> List[torch.Tensor]:
    """Compute per-token log probabilities via teacher forcing.

    For each (prompt, response) pair, concatenates prompt_ids and
    response_ids, forward-passes through the model, and extracts
    log p(y_t | y_{<t}, x) at each *response* token position.

    Processes one sample at a time to avoid padding complications.

    Args:
        model: Causal LM on GPU.
        tokenizer: Tokenizer.
        generated_results: Output from generate_responses().

    Returns:
        List of 1-D CPU tensors, one per sample. Each tensor has
        length equal to the number of response tokens.
    """
    model.eval()
    all_log_probs = []

    for result in tqdm(generated_results, desc="Computing log probs"):
        prompt_ids = result["prompt_ids"]
        response_ids = result["response_ids"]

        if len(response_ids) == 0:
            all_log_probs.append(torch.tensor([]))
            continue

        # Concatenate prompt + response
        full_ids = torch.cat([prompt_ids, response_ids]).unsqueeze(0).to(model.device)
        prompt_len = len(prompt_ids)

        logits = model(full_ids).logits  # (1, seq_len, vocab_size)

        # Standard causal LM shift: logits[t] predicts token[t+1]
        shift_logits = logits[:, :-1, :]
        shift_labels = full_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        token_log_probs = log_probs.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        # Response tokens start at position prompt_len in the full sequence.
        # After the shift, the log prob for position prompt_len is at index
        # (prompt_len - 1) in token_log_probs.
        resp_start = prompt_len - 1
        resp_lp = token_log_probs[0, resp_start:].cpu().float()

        all_log_probs.append(resp_lp)

    return all_log_probs


def compute_delta_log_p(
    log_probs_ft: List[torch.Tensor],
    log_probs_base: List[torch.Tensor],
) -> List[torch.Tensor]:
    """Compute per-token delta_log_p = log p_finetuned - log p_base.

    Handles length mismatches by truncating to the shorter sequence.

    Args:
        log_probs_ft: Per-sample token log probs from the fine-tuned model.
        log_probs_base: Per-sample token log probs from the base model.

    Returns:
        List of per-token delta_log_p tensors.
    """
    deltas = []
    for lp_ft, lp_base in zip(log_probs_ft, log_probs_base):
        min_len = min(len(lp_ft), len(lp_base))
        if min_len == 0:
            deltas.append(torch.tensor([]))
        else:
            deltas.append(lp_ft[:min_len] - lp_base[:min_len])
    return deltas


In [ ]:
# ============================================================================
# Critical Token Identification
# ============================================================================

def identify_critical_tokens_abs(
    delta_log_p_list: List[torch.Tensor],
    percentile: float = 90,
) -> Tuple[List[torch.Tensor], float]:
    """Identify critical tokens where |delta_log_p| exceeds a percentile threshold.

    This is the basic version used by JWCM v2: selects tokens with the
    largest absolute probability change, regardless of sign.

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        percentile: Percentile threshold (e.g. 90 = top 10% are critical).

    Returns:
        Tuple of (masks, threshold):
        - masks: List of boolean tensors (True = critical token)
        - threshold: Computed absolute threshold value
    """
    # Gather all |delta_log_p| values for global threshold
    all_abs = torch.cat([d.abs() for d in delta_log_p_list if len(d) > 0])
    threshold = torch.quantile(all_abs.float(), percentile / 100.0).item()

    masks = []
    for delta in delta_log_p_list:
        if len(delta) == 0:
            masks.append(torch.tensor([], dtype=torch.bool))
        else:
            masks.append(delta.abs() > threshold)

    total_tokens = sum(len(d) for d in delta_log_p_list)
    critical_count = sum(m.sum().item() for m in masks)
    print(f"|delta_log_p| threshold: {threshold:.4f}")
    print(f"Critical tokens: {critical_count}/{total_tokens} "
          f"({critical_count / max(total_tokens, 1) * 100:.1f}%)")

    return masks, threshold


def identify_critical_tokens_bidirectional(
    delta_log_p_list: List[torch.Tensor],
    percentile: float = 90,
) -> Tuple[List[torch.Tensor], List[torch.Tensor], float, float]:
    """Identify critical tokens with separate amplify (C+) and suppress (C-) sets.

    JWCM v3 uses this to handle RLVR's bidirectional updates:
    - C+ (amplify): tokens whose probability INCREASED (delta_log_p > +threshold)
    - C- (suppress): tokens whose probability DECREASED (delta_log_p < -threshold)

    The threshold is computed from the global distribution of delta_log_p
    (not |delta_log_p|) to allow different thresholds for + and -.

    For simplicity we use symmetric thresholds based on |delta_log_p|.

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        percentile: Percentile for |delta_log_p| threshold.

    Returns:
        Tuple of (masks_plus, masks_minus, threshold_plus, threshold_minus).
    """
    all_abs = torch.cat([d.abs() for d in delta_log_p_list if len(d) > 0])
    threshold = torch.quantile(all_abs.float(), percentile / 100.0).item()

    masks_plus = []
    masks_minus = []
    for delta in delta_log_p_list:
        if len(delta) == 0:
            masks_plus.append(torch.tensor([], dtype=torch.bool))
            masks_minus.append(torch.tensor([], dtype=torch.bool))
        else:
            masks_plus.append(delta > threshold)       # amplified tokens
            masks_minus.append(delta < -threshold)      # suppressed tokens

    total_tokens = sum(len(d) for d in delta_log_p_list)
    plus_count = sum(m.sum().item() for m in masks_plus)
    minus_count = sum(m.sum().item() for m in masks_minus)
    print(f"Threshold: +/-{threshold:.4f}")
    print(f"C+ (amplify):  {plus_count}/{total_tokens} ({plus_count / max(total_tokens, 1) * 100:.1f}%)")
    print(f"C- (suppress): {minus_count}/{total_tokens} ({minus_count / max(total_tokens, 1) * 100:.1f}%)")

    return masks_plus, masks_minus, threshold, threshold


def visualize_delta_log_p(
    delta_log_p_list: List[torch.Tensor],
    task_name: str,
    output_dir: Path,
):
    """Plot distribution of delta_log_p values for a task.

    Creates a histogram showing the distribution of token-level probability
    changes, highlighting the critical regions (tails).

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        task_name: Name for plot title (e.g. "IF" or "Math").
        output_dir: Directory to save the figure.
    """
    all_vals = torch.cat([d for d in delta_log_p_list if len(d) > 0]).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Histogram of delta_log_p
    axes[0].hist(all_vals, bins=200, density=True, alpha=0.7, color="steelblue")
    axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[0].set_xlabel("delta_log_p")
    axes[0].set_ylabel("Density")
    axes[0].set_title(f"{task_name}: Distribution of delta_log_p")

    # Histogram of |delta_log_p|
    axes[1].hist(np.abs(all_vals), bins=200, density=True, alpha=0.7, color="darkorange")
    p90 = np.percentile(np.abs(all_vals), CRITICAL_PERCENTILE)
    axes[1].axvline(p90, color="red", linestyle="--", linewidth=1,
                    label=f"p{CRITICAL_PERCENTILE} = {p90:.3f}")
    axes[1].set_xlabel("|delta_log_p|")
    axes[1].set_ylabel("Density")
    axes[1].set_title(f"{task_name}: Distribution of |delta_log_p|")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_dir / f"delta_log_p_distribution_{task_name.lower()}.png", dpi=150)
    plt.show()
    print(f"Saved {task_name} delta_log_p distribution plot.")


In [ ]:
# ============================================================================
# Task Vector Computation
# ============================================================================

def compute_task_vectors(
    base_model_id: str,
    finetuned_paths: Dict[str, str],
    dtype=torch.float16,
) -> Dict[str, Dict[str, torch.Tensor]]:
    """Compute task vectors: delta_tau = theta_finetuned - theta_base.

    Loads each model pair on CPU, computes the difference per parameter,
    and stores task vectors as CPU tensors in float32 for numerical
    stability during attribution and merging.

    Args:
        base_model_id: HuggingFace ID or path for the base model.
        finetuned_paths: Dict mapping task name -> model path.
        dtype: Load dtype (fp16 for memory efficiency).

    Returns:
        Dict[task_name, Dict[param_name, delta_tensor]].
    """
    print("Loading base model state dict...")
    base_model = load_model(base_model_id, dtype=dtype, device="cpu")
    base_sd = {k: v.float().clone() for k, v in base_model.state_dict().items()}

    task_vectors = {}
    for task_name, ft_path in finetuned_paths.items():
        print(f"Computing task vector for: {task_name}")
        ft_model = load_model(ft_path, dtype=dtype, device="cpu")
        ft_sd = ft_model.state_dict()

        tv = {}
        for name in base_sd:
            if name in ft_sd and ft_sd[name].shape == base_sd[name].shape:
                tv[name] = ft_sd[name].float() - base_sd[name]
            else:
                print(f"  Skipping {name}: not found or shape mismatch")
        task_vectors[task_name] = tv

        del ft_model, ft_sd
        gc.collect()
        print(f"  {task_name}: {len(tv)} parameters, "
              f"total norm = {sum(v.norm().item()**2 for v in tv.values())**0.5:.4f}")

    del base_model, base_sd
    gc.collect()
    return task_vectors


In [ ]:
# ============================================================================
# JWCM v3: Bidirectional Attribution Score Computation
# ============================================================================

def compute_attribution_scores_v3(
    base_model_id: str,
    tokenizer,
    generated_results: List[dict],
    masks_plus: List[torch.Tensor],
    masks_minus: List[torch.Tensor],
    task_vector: Dict[str, torch.Tensor],
    task_name: str,
    dtype=torch.float16,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    """Compute bidirectional per-parameter attribution scores for JWCM v3.

    Unlike v2 which uses a single critical set, v3 separates:
    - C+ (amplify): tokens where delta_log_p > +threshold
    - C- (suppress): tokens where delta_log_p < -threshold

    Two backward passes per sample:
    - loss_plus  = sum log p(y_t) for t in C+
    - loss_minus = sum log p(y_t) for t in C-

    Attribution scores:
    - S_j^+ = Sigma_samples |grad_plus_j * delta_theta_j|
    - S_j^- = Sigma_samples |grad_minus_j * delta_theta_j|

    The final S_j = S_j^+ + S_j^- captures both amplify and suppress signals.

    Why this matters for RLVR:
    If a parameter has gradient +0.5 from amplify tokens and -0.5 from
    suppress tokens, v2's aggregated gradient gives |+0.5 + (-0.5)| = 0
    (parameter looks unimportant). v3 gives |+0.5| + |-0.5| = 1.0
    (parameter is critical for both directions).

    Args:
        base_model_id: Base model path.
        tokenizer: Tokenizer.
        generated_results: Generated (prompt, response) pairs.
        masks_plus: Boolean masks for C+ tokens.
        masks_minus: Boolean masks for C- tokens.
        task_vector: Task vector dict.
        task_name: Name for progress bar.
        dtype: Model load dtype.

    Returns:
        Tuple of (S_plus, S_minus), each Dict[param_name, tensor].
    """
    print(f"\nComputing bidirectional attribution for {task_name}...")
    model = load_model(base_model_id, dtype=dtype, device="cuda")
    model.train()

    # Initialize accumulators
    S_plus = {}
    S_minus = {}
    for name, param in model.named_parameters():
        param.requires_grad_(True)
        if name in task_vector:
            S_plus[name] = torch.zeros_like(task_vector[name], dtype=torch.float32)
            S_minus[name] = torch.zeros_like(task_vector[name], dtype=torch.float32)

    n_plus_samples = 0
    n_minus_samples = 0

    for idx in tqdm(range(len(generated_results)), desc=f"Bidir Attribution ({task_name})"):
        m_plus = masks_plus[idx]
        m_minus = masks_minus[idx]

        has_plus = len(m_plus) > 0 and m_plus.sum().item() > 0
        has_minus = len(m_minus) > 0 and m_minus.sum().item() > 0

        if not has_plus and not has_minus:
            continue

        result = generated_results[idx]
        prompt_ids = result["prompt_ids"]
        response_ids = result["response_ids"]

        if len(response_ids) == 0:
            continue

        full_ids = torch.cat([prompt_ids, response_ids]).unsqueeze(0).cuda()
        prompt_len = len(prompt_ids)

        # Forward pass (shared for both C+ and C-)
        logits = model(full_ids).logits
        shift_logits = logits[:, :-1, :]
        shift_labels = full_ids[:, 1:]
        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        token_log_probs = log_probs.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        resp_start = prompt_len - 1
        resp_lp = token_log_probs[0, resp_start:]

        # --- C+ backward pass ---
        if has_plus:
            n_plus_samples += 1
            eff_mask_plus = m_plus[: len(resp_lp)].float().cuda()
            loss_plus = (resp_lp * eff_mask_plus).sum()
            loss_plus.backward(retain_graph=has_minus)

            for name, param in model.named_parameters():
                if param.grad is not None and name in S_plus:
                    grad_cpu = param.grad.detach().float().cpu()
                    S_plus[name] += (grad_cpu * task_vector[name]).abs()

            model.zero_grad()

        # --- C- backward pass ---
        if has_minus:
            n_minus_samples += 1
            eff_mask_minus = m_minus[: len(resp_lp)].float().cuda()
            loss_minus = (resp_lp * eff_mask_minus).sum()
            loss_minus.backward()

            for name, param in model.named_parameters():
                if param.grad is not None and name in S_minus:
                    grad_cpu = param.grad.detach().float().cpu()
                    S_minus[name] += (grad_cpu * task_vector[name]).abs()

            model.zero_grad()

        if idx % 64 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    total_plus = sum(s.sum().item() for s in S_plus.values())
    total_minus = sum(s.sum().item() for s in S_minus.values())
    print(f"  C+ samples: {n_plus_samples}, total mass: {total_plus:.6f}")
    print(f"  C- samples: {n_minus_samples}, total mass: {total_minus:.6f}")

    free_model(model, f"base model (bidir attribution {task_name})")
    return S_plus, S_minus


In [ ]:
# ============================================================================
# JWCM v3 Merge: Bidirectional Attribution-Weighted Merge (N-task scalable)
# ============================================================================

ARTIFACT_DIR = Path("merging_analysis/artifacts/jwcm_v3")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_MODEL_DIR_V3 = ARTIFACT_DIR / "merged_model"


def merge_jwcm_v3(
    base_model_id: str,
    task_vectors: Dict[str, Dict[str, torch.Tensor]],
    S_plus: Dict[str, Dict[str, torch.Tensor]],
    S_minus: Dict[str, Dict[str, torch.Tensor]],
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    """Apply JWCM v3 bidirectional attribution-weighted merge.

    For each task tau, the total importance is:
        S_j^tau = S_j^{tau,+} + S_j^{tau,-}

    This ensures that parameters critical for EITHER amplification
    or suppression receive high weight.

    Merge formula (identical to v2 but with bidirectional S):
        theta_merge_j = theta_base_j
            + sum_tau (S_j^tau * delta_theta_j^tau) / (sum_tau S_j^tau + eps)

    Why v3 > v2 for RLVR:
    If gradient from C+ and C- have opposite signs for parameter j,
    v2's aggregated S underestimates importance (cancellation).
    v3 takes |.| before summing, capturing both directions.

    Scales to N tasks: just add more entries to task_vectors, S_plus, S_minus.

    Args:
        base_model_id: Base model path.
        task_vectors: {task_name: {param_name: delta}}.
        S_plus: {task_name: {param_name: S^+}}.
        S_minus: {task_name: {param_name: S^-}}.
        eps: Division epsilon.

    Returns:
        Merged state dict.
    """
    print("Applying JWCM v3 bidirectional merge...")
    base_model = load_model(base_model_id, dtype=DTYPE, device="cpu")
    base_sd = {k: v.float().clone() for k, v in base_model.state_dict().items()}
    del base_model
    gc.collect()

    task_names = list(task_vectors.keys())
    merged_sd = {}

    for name in tqdm(base_sd, desc="Merging (v3)"):
        has_all = all(
            name in task_vectors[t]
            and name in S_plus[t]
            and name in S_minus[t]
            for t in task_names
        )

        if not has_all:
            merged_sd[name] = base_sd[name]
            continue

        numerator = torch.zeros_like(base_sd[name])
        denominator = torch.zeros_like(base_sd[name])

        for t in task_names:
            # Bidirectional total importance
            S_total_t = S_plus[t][name] + S_minus[t][name]
            delta_t = task_vectors[t][name]
            numerator += S_total_t * delta_t
            denominator += S_total_t

        merged_sd[name] = base_sd[name] + numerator / (denominator + eps)

    return merged_sd


def save_merged_model(
    merged_sd: Dict[str, torch.Tensor],
    base_model_id: str,
    output_dir: Path,
    dtype=torch.float16,
):
    """Save merged state dict as a HuggingFace model checkpoint."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Saving merged model to {output_dir}...")
    model = load_model(base_model_id, dtype=torch.float32, device="cpu")
    converted_sd = {k: v.to(dtype) for k, v in merged_sd.items()}
    model.load_state_dict(converted_sd, strict=False)
    model.save_pretrained(output_dir)

    tokenizer = load_tokenizer(base_model_id)
    tokenizer.save_pretrained(output_dir)

    del model, converted_sd
    gc.collect()
    print(f"Saved merged model + tokenizer to {output_dir}")


## Execution: Run the Full Pipeline


In [ ]:
# ============================================================================
# Step 1: Sample Validation Data (512 per benchmark)
# ============================================================================
val_data_dir = Path("merging_analysis/artifacts/jwcm_v3/validation_data")
val_data_dir.mkdir(parents=True, exist_ok=True)

val_if_path = val_data_dir / "val_if_512.parquet"
val_math_path = val_data_dir / "val_math_512.parquet"

if val_if_path.exists() and val_math_path.exists():
    print("Loading cached validation data...")
    val_if = pd.read_parquet(val_if_path)
    val_math = pd.read_parquet(val_math_path)
else:
    print("Sampling validation data...")
    val_if = sample_validation_data(IF_DATA_PATH, VAL_SAMPLES_PER_TASK, seed=SEED)
    val_math = sample_validation_data(MATH_DATA_PATH, VAL_SAMPLES_PER_TASK, seed=SEED)
    val_if.to_parquet(val_if_path)
    val_math.to_parquet(val_math_path)

prompts_if = extract_prompts(val_if)
prompts_math = extract_prompts(val_math)
print(f"IF validation samples: {len(prompts_if)}")
print(f"Math validation samples: {len(prompts_math)}")


In [ ]:
# ============================================================================
# Step 2: Generate Responses from Fine-tuned Models
# ============================================================================
gen_dir = Path("merging_analysis/artifacts/jwcm_v3/generated_responses")
gen_dir.mkdir(parents=True, exist_ok=True)

tokenizer = load_tokenizer(BASE_MODEL_ID)

# --- Generate IF responses ---
gen_if_path = gen_dir / "gen_if.pt"
if gen_if_path.exists():
    print("Loading cached IF responses...")
    gen_if = torch.load(gen_if_path, weights_only=False)
else:
    print("Generating IF responses...")
    if_model = load_model(IF_MODEL_PATH, dtype=DTYPE, device="cuda")
    gen_if = generate_responses(
        if_model, tokenizer, prompts_if,
        max_new_tokens=MAX_NEW_TOKENS, batch_size=GENERATION_BATCH_SIZE,
    )
    free_model(if_model, "IF model")
    torch.save(gen_if, gen_if_path)
print(f"IF responses: {len(gen_if)}")

# --- Generate Math responses ---
gen_math_path = gen_dir / "gen_math.pt"
if gen_math_path.exists():
    print("Loading cached Math responses...")
    gen_math = torch.load(gen_math_path, weights_only=False)
else:
    print("Generating Math responses...")
    math_model = load_model(MATH_MODEL_PATH, dtype=DTYPE, device="cuda")
    gen_math = generate_responses(
        math_model, tokenizer, prompts_math,
        max_new_tokens=MAX_NEW_TOKENS, batch_size=GENERATION_BATCH_SIZE,
    )
    free_model(math_model, "Math model")
    torch.save(gen_math, gen_math_path)
print(f"Math responses: {len(gen_math)}")

# Print sample responses
for task_name, gen_results in [("IF", gen_if), ("Math", gen_math)]:
    print(f"\n--- {task_name} Sample Response ---")
    r = gen_results[0]
    print(f"Response (first 200 chars): {r['response_text'][:200]}...")
    print(f"Prompt tokens: {len(r['prompt_ids'])}, Response tokens: {len(r['response_ids'])}")


In [ ]:
# ============================================================================
# Step 3: Compute delta_log_p and Identify Bidirectional Critical Tokens
# ============================================================================
dlp_dir = ARTIFACT_DIR / "delta_log_p"
dlp_dir.mkdir(parents=True, exist_ok=True)

dlp_if_path = dlp_dir / "delta_log_p_if.pt"
dlp_math_path = dlp_dir / "delta_log_p_math.pt"

if dlp_if_path.exists() and dlp_math_path.exists():
    print("Loading cached delta_log_p...")
    delta_lp_if = torch.load(dlp_if_path, weights_only=False)
    delta_lp_math = torch.load(dlp_math_path, weights_only=False)
else:
    # Compute log probs under fine-tuned models
    print("Computing log probs under IF model...")
    if_model = load_model(IF_MODEL_PATH, dtype=DTYPE, device="cuda")
    lp_if_ft = compute_token_log_probs(if_model, tokenizer, gen_if)
    free_model(if_model, "IF model (log probs)")

    print("Computing log probs under Math model...")
    math_model = load_model(MATH_MODEL_PATH, dtype=DTYPE, device="cuda")
    lp_math_ft = compute_token_log_probs(math_model, tokenizer, gen_math)
    free_model(math_model, "Math model (log probs)")

    # Compute log probs under base model
    print("Computing log probs under Base model...")
    base_model = load_model(BASE_MODEL_ID, dtype=DTYPE, device="cuda")
    lp_if_base = compute_token_log_probs(base_model, tokenizer, gen_if)
    lp_math_base = compute_token_log_probs(base_model, tokenizer, gen_math)
    free_model(base_model, "Base model (log probs)")

    delta_lp_if = compute_delta_log_p(lp_if_ft, lp_if_base)
    delta_lp_math = compute_delta_log_p(lp_math_ft, lp_math_base)

    torch.save(delta_lp_if, dlp_if_path)
    torch.save(delta_lp_math, dlp_math_path)

# Visualize distributions
crit_dir = ARTIFACT_DIR / "critical_token_analysis"
crit_dir.mkdir(parents=True, exist_ok=True)
visualize_delta_log_p(delta_lp_if, "IF", crit_dir)
visualize_delta_log_p(delta_lp_math, "Math", crit_dir)

# Identify bidirectional critical tokens (v3: separate C+ and C-)
print("\n--- IF Bidirectional Critical Tokens ---")
masks_if_plus, masks_if_minus, _, _ = identify_critical_tokens_bidirectional(
    delta_lp_if, percentile=CRITICAL_PERCENTILE
)
print("\n--- Math Bidirectional Critical Tokens ---")
masks_math_plus, masks_math_minus, _, _ = identify_critical_tokens_bidirectional(
    delta_lp_math, percentile=CRITICAL_PERCENTILE
)


In [ ]:
# ============================================================================
# Step 4: Compute Task Vectors and Bidirectional Attribution Scores
# ============================================================================
attr_dir = ARTIFACT_DIR / "attribution_scores"
attr_dir.mkdir(parents=True, exist_ok=True)

tv_path = attr_dir / "task_vectors.pt"
attr_if_plus_path = attr_dir / "S_if_plus.pt"
attr_if_minus_path = attr_dir / "S_if_minus.pt"
attr_math_plus_path = attr_dir / "S_math_plus.pt"
attr_math_minus_path = attr_dir / "S_math_minus.pt"

# Compute task vectors
if tv_path.exists():
    print("Loading cached task vectors...")
    task_vectors = torch.load(tv_path, weights_only=False)
else:
    task_vectors = compute_task_vectors(
        BASE_MODEL_ID,
        {"IF": IF_MODEL_PATH, "Math": MATH_MODEL_PATH},
        dtype=DTYPE,
    )
    torch.save(task_vectors, tv_path)

# Compute bidirectional attribution scores for IF
if attr_if_plus_path.exists() and attr_if_minus_path.exists():
    print("Loading cached IF bidirectional attribution scores...")
    S_if_plus = torch.load(attr_if_plus_path, weights_only=False)
    S_if_minus = torch.load(attr_if_minus_path, weights_only=False)
else:
    S_if_plus, S_if_minus = compute_attribution_scores_v3(
        BASE_MODEL_ID, tokenizer, gen_if,
        masks_if_plus, masks_if_minus,
        task_vectors["IF"], "IF", dtype=DTYPE,
    )
    torch.save(S_if_plus, attr_if_plus_path)
    torch.save(S_if_minus, attr_if_minus_path)

# Compute bidirectional attribution scores for Math
if attr_math_plus_path.exists() and attr_math_minus_path.exists():
    print("Loading cached Math bidirectional attribution scores...")
    S_math_plus = torch.load(attr_math_plus_path, weights_only=False)
    S_math_minus = torch.load(attr_math_minus_path, weights_only=False)
else:
    S_math_plus, S_math_minus = compute_attribution_scores_v3(
        BASE_MODEL_ID, tokenizer, gen_math,
        masks_math_plus, masks_math_minus,
        task_vectors["Math"], "Math", dtype=DTYPE,
    )
    torch.save(S_math_plus, attr_math_plus_path)
    torch.save(S_math_minus, attr_math_minus_path)

S_plus_all = {"IF": S_if_plus, "Math": S_math_plus}
S_minus_all = {"IF": S_if_minus, "Math": S_math_minus}

# Print summary
for task_name in ["IF", "Math"]:
    total_plus = sum(s.sum().item() for s in S_plus_all[task_name].values())
    total_minus = sum(s.sum().item() for s in S_minus_all[task_name].values())
    print(f"{task_name}: S+ mass={total_plus:.4f}, S- mass={total_minus:.4f}, "
          f"ratio S-/S+={total_minus / max(total_plus, 1e-12):.4f}")


In [ ]:
# ============================================================================
# Step 5: Apply JWCM v3 Bidirectional Merge + Save
# ============================================================================

merged_sd_v3 = merge_jwcm_v3(
    BASE_MODEL_ID, task_vectors, S_plus_all, S_minus_all,
)
save_merged_model(merged_sd_v3, BASE_MODEL_ID, MERGED_MODEL_DIR_V3, dtype=DTYPE)
del merged_sd_v3
gc.collect()

print("\n" + "=" * 60)
print("JWCM v3 merge complete!")
print(f"  v3 model: {MERGED_MODEL_DIR_V3}")
print("=" * 60)


## Diagnostics: Bidirectional Analysis


In [ ]:
# ============================================================================
# Step 6: Diagnostics - Compare v3 Bidirectional vs Unidirectional
# ============================================================================

import re

def plot_bidirectional_attribution_comparison(
    S_plus: Dict[str, Dict[str, torch.Tensor]],
    S_minus: Dict[str, Dict[str, torch.Tensor]],
    output_dir: Path,
):
    """Compare amplify vs suppress attribution mass per layer.

    For RLVR models, this reveals whether the model's updates are
    primarily amplification (increasing good tokens) or suppression
    (decreasing bad tokens) at each layer.

    Args:
        S_plus: {task_name: {param_name: S_plus_tensor}}.
        S_minus: {task_name: {param_name: S_minus_tensor}}.
        output_dir: Directory to save figure.
    """
    task_names = list(S_plus.keys())

    fig, axes = plt.subplots(1, len(task_names), figsize=(8 * len(task_names), 6))
    if len(task_names) == 1:
        axes = [axes]

    for ax, t in zip(axes, task_names):
        layer_plus = defaultdict(float)
        layer_minus = defaultdict(float)

        for name in S_plus[t]:
            match = re.search(r"model\.layers\.(\d+)\.", name)
            if match:
                idx = int(match.group(1))
            elif "embed" in name:
                idx = -1
            elif "lm_head" in name:
                idx = 999
            else:
                continue
            layer_plus[idx] += S_plus[t][name].sum().item()
            layer_minus[idx] += S_minus[t][name].sum().item()

        layers = sorted(set(layer_plus.keys()) | set(layer_minus.keys()))
        labels = []
        vals_p = []
        vals_m = []
        for l in layers:
            labels.append("emb" if l == -1 else ("lm_h" if l == 999 else str(l)))
            vals_p.append(layer_plus.get(l, 0))
            vals_m.append(layer_minus.get(l, 0))

        x = np.arange(len(layers))
        width = 0.35
        ax.bar(x - width / 2, vals_p, width, label="S+ (amplify)", color="steelblue")
        ax.bar(x + width / 2, vals_m, width, label="S- (suppress)", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Attribution Mass")
        ax.set_title(f"{t}: Amplify vs Suppress Attribution")
        ax.legend()
        ax.set_yscale("log")

    fig.tight_layout()
    fig.savefig(output_dir / "bidirectional_attribution_comparison.png", dpi=150)
    plt.show()

plot_bidirectional_attribution_comparison(S_plus_all, S_minus_all, ARTIFACT_DIR)

# Quantify the difference between v2-style and v3-style attribution
print("\n--- v2 vs v3 Attribution Comparison ---")
for task_name in ["IF", "Math"]:
    # v2-style: |sum of grads| * |delta|
    # v3-style: |grad+| * |delta| + |grad-| * |delta|
    # The ratio v3/v2 > 1 indicates bidirectional cancellation in v2
    v3_mass = sum(
        (S_plus_all[task_name][n] + S_minus_all[task_name][n]).sum().item()
        for n in S_plus_all[task_name]
    )
    print(f"  {task_name}: v3 total mass = {v3_mass:.4f}")
    print(f"  (Compare with v2 mass from the v2 notebook to see cancellation effect)")
